In [1]:
from pathlib import Path
import pandas as pd


def encontrar_raiz():
    """Localiza PRE_12_supply_chain desde el directorio de ejecución."""
    actual = Path.cwd().resolve()

    for carpeta in (actual, *actual.parents):
        if (carpeta / "data" / "supply_chain.csv").is_file():
            return carpeta

        candidata = carpeta / "PRE_12_supply_chain"
        if (candidata / "data" / "supply_chain.csv").is_file():
            return candidata

    raise FileNotFoundError(
        "No se encontró data/supply_chain.csv. "
        "Ejecuta el notebook desde PRE_12_supply_chain o notebooks."
    )


def convertir_numero(serie):
    """
    Convierte valores numéricos.
    Los textos como 'Freight Included in Commodity Cost'
    quedan como datos faltantes, no como ceros.
    """
    texto = serie.astype("string").str.strip()
    texto = texto.str.replace(",", "", regex=False)
    texto = texto.str.replace("$", "", regex=False)

    return pd.to_numeric(texto, errors="coerce")


def suma_disponible(serie):
    """Evita reportar cero cuando todos los valores son desconocidos."""
    return serie.sum(min_count=1)


def resumir(datos, grupos):
    """Calcula indicadores por una o varias categorías."""
    resultado = (
        datos.groupby(grupos, dropna=False)
        .agg(
            line_count=("ID", "size"),
            total_quantity=("Line Item Quantity", suma_disponible),
            total_item_value_usd=("Line Item Value", suma_disponible),
            total_freight_usd=("Freight Cost (USD)", suma_disponible),
            freight_numeric_count=("Freight Cost (USD)", "count"),
            total_weight_kg=("Weight (Kilograms)", suma_disponible),
            total_insurance_usd=("Line Item Insurance (USD)", suma_disponible),
            evaluated_delivery_count=("delay_days", "count"),
            late_line_count=("is_late", "sum"),
            on_time_rate=("is_on_time", "mean"),
            average_delay_days=("delay_days", "mean"),
        )
        .reset_index()
    )

    resultado["on_time_rate_pct"] = resultado.pop("on_time_rate") * 100

    return resultado


def main():
    raiz = encontrar_raiz()
    salida = raiz / "submission"
    salida.mkdir(parents=True, exist_ok=True)

    # 1. Leer y limpiar los datos.
    df = pd.read_csv(
        raiz / "data" / "supply_chain.csv",
        encoding="utf-8-sig",
        low_memory=False,
    )

    df.columns = df.columns.str.strip()

    columnas_requeridas = [
        "ID",
        "Country",
        "Shipment Mode",
        "Scheduled Delivery Date",
        "Delivered to Client Date",
        "Line Item Quantity",
        "Line Item Value",
        "Weight (Kilograms)",
        "Freight Cost (USD)",
        "Line Item Insurance (USD)",
    ]

    faltantes = [
        columna
        for columna in columnas_requeridas
        if columna not in df.columns
    ]

    if faltantes:
        raise ValueError(f"Faltan columnas requeridas: {faltantes}")

    for columna in ["Country", "Shipment Mode"]:
        df[columna] = (
            df[columna]
            .astype("string")
            .str.strip()
            .replace("", pd.NA)
            .fillna("Unknown")
        )

    columnas_numericas = [
        "Line Item Quantity",
        "Line Item Value",
        "Weight (Kilograms)",
        "Freight Cost (USD)",
        "Line Item Insurance (USD)",
    ]

    for columna in columnas_numericas:
        df[columna] = convertir_numero(df[columna])

    # Formato de las fechas mostradas: 2-Jun-06.
    for columna in [
        "Scheduled Delivery Date",
        "Delivered to Client Date",
    ]:
        df[columna] = pd.to_datetime(
            df[columna].astype("string").str.strip(),
            format="%d-%b-%y",
            errors="coerce",
        )

    # 2. Calcular puntualidad.
    # Positivo: entrega tardía.
    # Cero: entrega en la fecha prevista.
    # Negativo: entrega anticipada.
    df["delay_days"] = (
        df["Delivered to Client Date"] - df["Scheduled Delivery Date"]
    ).dt.days

    fechas_validas = df["delay_days"].notna()

    # Las filas sin fechas válidas no cuentan como puntuales ni tardías.
    df["is_late"] = (
        df["delay_days"].gt(0).astype("Float64").where(fechas_validas)
    )
    df["is_on_time"] = (
        df["delay_days"].le(0).astype("Float64").where(fechas_validas)
    )

    # Mes de entrega real.
    df["delivery_month"] = (
        df["Delivered to Client Date"].dt.to_period("M").astype("string")
    )

    # 3. Resúmenes por país, transporte y combinación de ambos.
    country_summary = resumir(df, ["Country"])
    mode_summary = resumir(df, ["Shipment Mode"])
    country_mode_summary = resumir(df, ["Country", "Shipment Mode"])

    # Solo se incluyen en el resumen mensual las fechas reales válidas.
    monthly_summary = resumir(
        df.loc[df["Delivered to Client Date"].notna()],
        ["delivery_month"],
    ).sort_values("delivery_month")

    # 4. Estadísticas de fletes por medio de transporte.
    freight_by_mode = (
        df.groupby("Shipment Mode", dropna=False)
        .agg(
            line_count=("ID", "size"),
            freight_numeric_count=("Freight Cost (USD)", "count"),
            total_freight_usd=("Freight Cost (USD)", suma_disponible),
            average_freight_usd=("Freight Cost (USD)", "mean"),
            median_freight_usd=("Freight Cost (USD)", "median"),
            min_freight_usd=("Freight Cost (USD)", "min"),
            max_freight_usd=("Freight Cost (USD)", "max"),
        )
        .reset_index()
    )

    freight_by_mode["freight_missing_or_text_count"] = (
        freight_by_mode["line_count"]
        - freight_by_mode["freight_numeric_count"]
    )

    # 5. Indicadores generales.
    overall_kpis = pd.DataFrame(
        [{
            "total_lines": len(df),
            "country_count": df.loc[
                df["Country"].ne("Unknown"), "Country"
            ].nunique(),
            "shipment_mode_count": df.loc[
                df["Shipment Mode"].ne("Unknown"), "Shipment Mode"
            ].nunique(),
            "total_quantity": suma_disponible(df["Line Item Quantity"]),
            "total_item_value_usd": suma_disponible(df["Line Item Value"]),
            "total_freight_usd": suma_disponible(df["Freight Cost (USD)"]),
            "total_weight_kg": suma_disponible(df["Weight (Kilograms)"]),
            "total_insurance_usd": suma_disponible(
                df["Line Item Insurance (USD)"]
            ),
            "evaluated_delivery_count": int(fechas_validas.sum()),
            "unevaluated_delivery_count": int((~fechas_validas).sum()),
            "late_line_count": int(df["is_late"].sum()),
            "on_time_rate_pct": df["is_on_time"].mean() * 100,
            "average_delay_days": df["delay_days"].mean(),
            "freight_numeric_count": df["Freight Cost (USD)"].count(),
        }]
    )

    # 6. Prioridades.
    # Criterio propuesto, porque el test no define uno:
    # grupos con retrasos, ordenados por cantidad de filas tardías;
    # en caso de empate, mayor retraso promedio y mayor valor de bienes.
    def crear_prioridades(resumen):
        prioridades = (
            resumen.loc[resumen["late_line_count"] > 0]
            .sort_values(
                [
                    "late_line_count",
                    "average_delay_days",
                    "total_item_value_usd",
                ],
                ascending=[False, False, False],
                kind="stable",
            )
            .reset_index(drop=True)
        )

        prioridades.insert(
            0, "priority_rank", range(1, len(prioridades) + 1)
        )

        return prioridades

    priority_countries = crear_prioridades(country_summary)

    # Se interpreta "segmento" como la combinación país–transporte.
    priority_segments = crear_prioridades(country_mode_summary)

    # 7. Guardar los ocho archivos solicitados.
    resultados = {
        "country_mode_summary.csv": country_mode_summary,
        "country_summary.csv": country_summary,
        "freight_by_mode.csv": freight_by_mode,
        "mode_summary.csv": mode_summary,
        "monthly_summary.csv": monthly_summary,
        "overall_kpis.csv": overall_kpis,
        "priority_countries.csv": priority_countries,
        "priority_segments.csv": priority_segments,
    }

    for nombre, tabla in resultados.items():
        tabla.to_csv(
            salida / nombre,
            index=False,
            encoding="utf-8",
            float_format="%.4f",
        )
        print(f"Generado: submission/{nombre} ({len(tabla)} filas)")

    return resultados


resultados = main()

Generado: submission/country_mode_summary.csv (97 filas)
Generado: submission/country_summary.csv (43 filas)
Generado: submission/freight_by_mode.csv (5 filas)
Generado: submission/mode_summary.csv (5 filas)
Generado: submission/monthly_summary.csv (113 filas)
Generado: submission/overall_kpis.csv (1 filas)
Generado: submission/priority_countries.csv (28 filas)
Generado: submission/priority_segments.csv (43 filas)
